# HybridSmoother

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridSmoother.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam

`HybridSmoother` implements an incremental fixed-lag smoother for hybrid systems. Unlike a full ISAM approach which keeps the entire history, a fixed-lag smoother typically marginalizes out older variables to maintain a bounded-size active window.

*CURRENT STATUS (based on provided header):* The header `HybridSmoother.h` exists, but the implementation details provided are minimal. It seems designed to manage a `HybridBayesNet` posterior and potentially update it incrementally. It includes features for:
    *   Storing all factors (`HybridNonlinearFactorGraph`).
    *   Maintaining a linearization point (`Values`).
    *   Holding the current posterior (`HybridBayesNet`).
    *   Optionally removing "dead modes" based on a marginal probability threshold (`marginalThreshold_`).
    *   An `update` method to incorporate new factors.
    *   A `relinearize` method.

This documentation provides a conceptual outline based on the header. Specific usage may require consulting detailed implementation or examples if available. It appears geared towards maintaining and updating a `HybridBayesNet` representation rather than a `HybridBayesTree` like `HybridNonlinearISAM`.

In [ ]:
import gtsam
import numpy as np
from gtsam import symbol_shorthand

# Smoother class
from gtsam import HybridSmoother

# Factors & Graphs
from gtsam import HybridNonlinearFactorGraph, HybridGaussianFactorGraph
from gtsam import PriorFactorPose2, BetweenFactorPose2, Pose2, Point3
from gtsam import DecisionTreeFactor, DiscreteKey, HybridNonlinearFactor

# Values & Results
from gtsam import Values, DiscreteValues, HybridValues, Ordering
from gtsam import HybridBayesNet

X = symbol_shorthand.X # Continuous Key (Pose2)
D = symbol_shorthand.D # Discrete Key

## Initialization

Constructed optionally with a `marginalThreshold` for dead mode removal. It starts with an empty internal state.

In [ ]:
# Initialize without dead mode removal
smoother1 = gtsam.HybridSmoother()
print("Initialized Smoother 1 (no threshold)")

# Initialize with dead mode removal threshold
threshold = 0.99
smoother2 = gtsam.HybridSmoother(marginalThreshold=threshold)
print(f"Initialized Smoother 2 (threshold={threshold})")
# print(f" Smoother 2 initial fixed values: {smoother2.fixedValues()}")

## Update Steps

The `update` method incorporates new nonlinear factors and initial estimates. It likely performs linearization, potentially updates the internal `HybridBayesNet` posterior (possibly by eliminating variables related to the new factors against the current posterior), and updates the linearization point. Pruning based on `maxNrLeaves` and `marginalThreshold` might occur during this step.

In [ ]:
smoother = gtsam.HybridSmoother(marginalThreshold=0.99)

# --- Initial Step (Pose 0 and Mode Prior) ---
step0_graph = gtsam.HybridNonlinearFactorGraph()
step0_values = gtsam.Values()
dk0 = DiscreteKey(D(0), 2)

prior_noise = gtsam.noiseModel.Diagonal.Sigmas(Point3(0.1, 0.1, 0.05))
step0_graph.add(PriorFactorPose2(X(0), Pose2(0, 0, 0), prior_noise))
step0_values.insert(X(0), Pose2(0.0, 0.0, 0.0)) # Initial estimate for X0

step0_graph.add(DecisionTreeFactor([dk0], "0.995 0.005")) # High prior on D0=0

print("--- Update 0 ---")
# maxNrLeaves might control complexity of resulting discrete factors/conditionals
smoother.update(step0_graph, step0_values, maxNrLeaves=10)
print("Smoother state after update 0:")
print(f" Lin Point Size: {smoother.linearizationPoint().size()}")
# print(f" Factors Size: {smoother.allFactors().size()}") # Might track factors
print(" Posterior HybridBayesNet:")
smoother.hybridBayesNet().print()
print(f" Fixed Values: {smoother.fixedValues()}") # D(0) might become fixed=0


# --- Second Step (Pose 1 and Hybrid Odometry) ---
step1_graph = gtsam.HybridNonlinearFactorGraph()
step1_values = gtsam.Values()

noise0 = gtsam.noiseModel.Diagonal.Sigmas(Point3(0.1, 0.1, np.radians(1)))
odom0 = BetweenFactorPose2(X(0), X(1), Pose2(1.0, 0, 0), noise0)
noise1 = gtsam.noiseModel.Diagonal.Sigmas(Point3(0.5, 0.5, np.radians(10)))
odom1 = BetweenFactorPose2(X(0), X(1), Pose2(1.0, 0, 0), noise1)
hybrid_odom = HybridNonlinearFactor(dk0, [odom0, odom1])
step1_graph.add(hybrid_odom)

x0_estimate = smoother.linearizationPoint().atPose2(X(0))
x1_initial_guess = x0_estimate.compose(Pose2(1.0, 0, 0))
step1_values.insert(X(1), x1_initial_guess)

print("\n--- Update 1 ---")
smoother.update(step1_graph, step1_values, maxNrLeaves=10)
print("Smoother state after update 1:")
print(f" Lin Point Size: {smoother.linearizationPoint().size()}")
print(" Posterior HybridBayesNet:")
smoother.hybridBayesNet().print()
print(f" Fixed Values: {smoother.fixedValues()}") # D(0) likely still fixed=0

## Accessing State and Optimization

Allows accessing the current linearization point, the posterior `HybridBayesNet`, and optimizing the posterior.

In [ ]:
# Get the current linearization point
lin_point = smoother.linearizationPoint()
print("\nCurrent Linearization Point:")
lin_point.print()

# Get the posterior Bayes Net
posterior_hbn = smoother.hybridBayesNet()
print("\nCurrent Posterior HybridBayesNet:")
posterior_hbn.print()

# Get fixed values determined during updates
fixed_vals = smoother.fixedValues()
print(f"\nCurrent Fixed Values: {fixed_vals}")

# Optimize the current posterior Bayes Net (may implicitly use fixed_vals)
map_solution = smoother.optimize()
print("\nMAP Solution from Smoother:")
map_solution.print()
# Note: The solution should respect the fixed values.
if D(0) in fixed_vals:
     print(f" Solution D(0): {map_solution.discrete().at(D(0))}, Fixed D(0): {fixed_vals.at(D(0))}")

## Relinearization

The `relinearize` method likely rebuilds the posterior `HybridBayesNet` by relinearizing all stored factors (`allFactors_`) around the current linearization point.

In [ ]:
print("\nRelinearizing...")
# This might be computationally expensive as it involves all factors
try:
    smoother.relinearize()
    print("Relinearization complete. Posterior HybridBayesNet:")
    smoother.hybridBayesNet().print()
    # Optimize again after relinearization
    map_solution_relinearized = smoother.optimize()
    print("\nMAP Solution after relinearization:")
    map_solution_relinearized.print()
except Exception as e:
    print(f"Relinearization failed: {e}")